# 🫵 Can Your Chunks Stand Alone? - Exercise

**An early RAG exercise: extract, clean, chunk, and inspect evidence.**

A colleague is preparing a documentation assistant for developers using the Census Data API. Your job is to prepare the source material so a later retriever has usable evidence to find. A chunk that contains a rule but loses its exception can be actively misleading.

Complete the five code sections marked **Your code** and the written response prompts. Supplied helpers are documented so you can explore how they work; you do not need to implement them.

Run the notebook in order. An unfinished TODO raises `NotImplementedError` deliberately: complete that section, rerun its cell, and then continue with the supplied checks.

**Prerequisite:** Demo 2 — Extraction, Cleanup and Chunking. This exercise reuses `PyMuPDFLoader`, `Document`, and `RecursiveCharacterTextSplitter` from `rag-demo-2-extract-normalize-chunk.ipynb`.

## Your mission


Complete five steps. Each step produces a result you can inspect before continuing.

| Step | Deliverable |
|---|---|
| 1. Inspect | Three concrete extraction observations |
| 2. Clean carefully | Cleaned pages with preservation checks |
| 3. Create a baseline | Chunks, source pages, and two boundary observations |
| 4. Compare | Three configurations and an evidence audit |
| 5. Repair and defend | One boundary repair, recheck, and a short decision note |

**Completion criteria:** preserve the source's meaning and traceability, show a concrete boundary failure, compare alternatives fairly, and justify one improvement. There is no universally correct chunk size and no requirement to make every audit case pass.


## The source PDF


We use the **U.S. Census Bureau, Census Data API User Guide, version 1.6 (2019)**. The complete, unmodified 20-page PDF is included in `assets/census_api_user_guide_v1_6.pdf`. We process only **PDF viewer pages 9-15**, which are **printed pages 5-11**.

The selection contains API query syntax, lists, limits, a numbered procedure, and a sentence that continues across a page boundary. It is a real developer document, not an artificially damaged teaching fixture. The full document is included so you can check surrounding context.

[Original PDF](https://www2.census.gov/api-documentation/api-guide.pdf)

**Read the document as a historical snapshot.** Questions in this exercise ask what this edition says, not what the service currently supports. You do not register for a key, visit the old example endpoints, or make API calls.

Open the supplied PDF in your PDF viewer. Page numbers in our code are explicitly named: `pdf_page` is the one-based viewer position; `printed_page` is the number printed on the page. The loader's original `page` field is zero-based.

## Setup

Keep this notebook beside the `assets` folder containing the supplied PDF. Use the same Python environment as Demo 2. Run the installation cell only if those packages are missing, then restart the kernel. Work through the TODOs in order; package installation is the only network-dependent setup.

We use a small document-specific cleaner built with ordinary Python. Its purpose is to make preservation decisions visible. It is not a general PDF reconstruction tool.

In [ ]:
# Uncomment only if needed in your notebook environment.
# !pip install pymupdf langchain-community langchain-text-splitters

In [ ]:
from typing import Any, Final
from pathlib import Path
import re
import json
import hashlib
from html import escape

from IPython.display import display, HTML

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
PDF_PATH: Final[Path] = Path("assets/census_api_user_guide_v1_6.pdf")
PDF_URL: Final[str] = "https://www2.census.gov/api-documentation/api-guide.pdf"
EXPECTED_SHA256: Final[str] = "b0ed1ca9acaaa04073692a93f044cd2a94bea3842f5dbdbbc7d795b39846c241"

if not PDF_PATH.is_file():
    raise FileNotFoundError("Extract the exercise ZIP and keep assets beside the notebook.")
if hashlib.sha256(PDF_PATH.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError("This exercise was prepared for the supplied PDF snapshot; use that copy.")

In [ ]:
# Display helper: accept ordinary dictionaries without requiring a table library.
def show_table(rows: list[dict[str, Any]]) -> None:
    """Display record dictionaries as an HTML table, escaping values as plain text."""
    if not rows:
        print("No rows.")
        return
    columns = list(rows[0])
    # Escape source text so symbols such as < and & are displayed literally.
    head = "".join(f"<th>{escape(str(k))}</th>" for k in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{escape(str(row.get(k, '')))}</td>" for k in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

## Step 1 - Inspect before cleaning

**Purpose:** distinguish extraction artifacts from meaningful content.

**Your assignment:**

1. Load the PDF with `PyMuPDFLoader` and keep viewer pages 9-15.
2. Print the text of viewer pages 11 and 13 and compare them with the PDF.
3. Record three observations: one removable artifact, one feature that must survive, and one problem that whitespace cleanup cannot solve.

**Expected result:** seven page documents, correct page labels, and specific observations grounded in the visible PDF.

**Connection to Demo 2:** the loader extracts text and metadata; successful extraction does not prove that reading order, paragraph structure, or evidence boundaries are correct.


### Your code

In [ ]:
# TODO - Step 1.
# 1. Load PDF_PATH with PyMuPDFLoader in page mode.
# 2. Select viewer pages 9-15 and create new Document objects.
# 3. Keep the extracted text and set source, pdf_page, and printed_page metadata.
#    pdf_page is one-based; printed_page is the number visible on the PDF page.
all_pages: list[Document]
selected_pages: list[Document] = []

# YOUR CODE HERE
raise NotImplementedError("Complete Step 1: load and label the selected pages.")


In [ ]:
assert len(selected_pages) == 7

show_table([{
    **d.metadata, "characters": len(d.page_content)
} for d in selected_pages])

In [ ]:
raw_by_page: dict[int, Document] = {
    d.metadata["pdf_page"]: d 
    for d in selected_pages
}

for page_number in (11, 13):
    print(f"\nPDF viewer page {page_number} (printed {page_number - 4})\n")
    print(raw_by_page[page_number].page_content)

### Your observations


Compare the extracted text with viewer pages 11 and 13. For each observation, identify the page and the specific text or layout feature that supports it.

- **A removable artifact:** *Your response here.*
- **A feature that must survive cleanup, and why:** *Your response here.*
- **A problem that whitespace cleanup cannot solve:** *Your response here.*


## Step 2 - Clean without erasing structure

**Purpose:** remove known page furniture and improve readability while preserving evidence.

**Your assignment:** complete `clean_page_text(text, printed_page)` so it:

1. Removes only the exact repeated guide title and the expected page-number line at the start.
2. Collapses repeated spaces and tabs within each line.
3. Reattaches an isolated bullet/number marker to the following text line, using the supplied helper.
4. Reflows wrapped prose with the supplied helper, retaining paragraph, heading, list, URL, and JSON-line boundaries.
5. Returns a new string; keep the original extraction available for comparison.

**Expected result:** before/after examples plus passing checks for page furniture, API syntax, numbers, procedure markers, and idempotence (cleaning twice changes nothing).

**Design choice:** we deliberately do not lowercase text, delete all punctuation, remove every standalone number, or blindly concatenate broken URLs. Those changes can alter identifiers or meaning. This cleaner handles the known layout conservatively; it does not recover every visual layout cue.


### Line-handling helpers - supplied

These helpers are infrastructure. `attach_markers` fixes detached list markers; `reflow_lines` joins ordinary wrapped prose, while preserving recognizable structural starts. Its heading patterns are specific to this selection. Students do not need to write a generic layout detector or learn regular expressions for this task.

A URL that is itself split across lines still needs inspection. Preserving its characters is different from certifying that it is executable.


In [ ]:
def attach_markers(lines: list[str]) -> list[str]:
    """Attach a detached list marker to its next nonempty line, preserving other text."""
    result: list[str] = []
    pending: str | None = None
    for line in lines:
        # A detached marker must survive until its following text arrives.
        if pending is not None and line:
            result.append(pending + " " + line)
            pending = None
        elif re.fullmatch(r"(?:\d+\.|[•▪o])", line):
            pending = line
        else:
            result.append(line)
    if pending is not None:
        result.append(pending)
    return result


def structural_start(line: str) -> bool:
    """Recognize likely block starts in this PDF; this is a layout-specific heuristic."""
    return bool(
        re.match(r"(?:\d+(?:\.\d+)+\s|\d+\.\s|[•▪o]\s|https?://|\[|\]|\{|\})", line)
        or line.isupper()
        or line.startswith(("Variables", "Time (", "Geography", "Examples", "Figure "))
    )


def reflow_lines(lines: list[str]) -> str:
    """Join wrapped prose while keeping detected headings, lists, and code lines apart."""
    blocks: list[str] = []
    pending: list[str] = []
    def flush() -> None:
        """Finish the current text block and clear the shared line buffer."""
        if pending:
            blocks.append(" ".join(pending))
            pending.clear()
    for line in lines:
        if not line:
            flush()
        elif structural_start(line):
            flush()
            pending.append(line)
            # Headings are their own blocks; numbered procedure items may wrap.
            heading = (bool(re.match(r"\d+(?:\.\d+)+\s", line))
                       or line.isupper()
                       or line.startswith(("Variables", "Time (", "Geography", "Examples", "Figure ")))
            if heading:
                flush()
            # Keep complete URL / JSON lines separate from surrounding prose.
            if line.startswith(("http://", "https://", "[", "]", "{", "}")):
                flush()
        else:
            pending.append(line)
    flush()
    return "\n\n".join(blocks)

### Your code

In [ ]:
# TODO - Step 2.
def clean_page_text(text: str, printed_page: int) -> str:
    """Remove verified page furniture and reflow text without changing source characters."""
    # TODO 1: Normalize spaces and tabs within each line while retaining line boundaries.
    # TODO 2: Remove the exact guide-title prefix and expected printed-page line only.
    # TODO 3: Use attach_markers and reflow_lines, then return the cleaned string.
    # YOUR CODE HERE
    raise NotImplementedError("Step 2: implement clean_page_text.")

In [ ]:
clean_pages: list[Document] = [
    Document(
        page_content=clean_page_text(d.page_content, d.metadata["printed_page"]),
        metadata=dict(d.metadata),
    ) 
    for d in selected_pages
]

clean_by_page: dict[int, Document] = {
    d.metadata["pdf_page"]: d 
    for d in clean_pages
}
print(clean_by_page[13].page_content[:1250])

### Preservation checks - supplied

The first check compares all non-whitespace characters after removing the verified page prefix. It catches lost digits, punctuation, negation, or list markers across the full selection. The other checks make a few important invariants easy to read. These are preservation checks, not proof that the result reproduces the PDF's visual reading order.


In [1]:
def without_furniture(text: str, printed_page: int) -> str:
    """Remove only the verified header prefix to create a preservation-test reference."""
    lines = [line.strip() for line in text.splitlines()]
    while lines and not lines[0]:
        lines.pop(0)
    if lines and lines[0] == "Census Data API User Guide":
        lines.pop(0)
        while lines and not lines[0]:
            lines.pop(0)
        if lines and lines[0] == str(printed_page):
            lines.pop(0)
    return "\n".join(lines)

In [ ]:
for raw, cleaned in zip(selected_pages, clean_pages):
    original_body = without_furniture(raw.page_content, raw.metadata["printed_page"])
    assert re.sub(r"\s", "", original_body) == re.sub(r"\s", "", cleaned.page_content)
    assert not cleaned.page_content.startswith("Census Data API User Guide")
    assert clean_page_text(cleaned.page_content, cleaned.metadata["printed_page"]) == cleaned.page_content

assert "&time=from+1997+to+2012" in clean_by_page[12].page_content
assert "&PAYANN=400000:500000" in clean_by_page[11].page_content
assert all(value in clean_by_page[13].page_content for value in ("50", "500", "per day", "IP"))
assert all(f"{n}. " in clean_by_page[13].page_content for n in range(1, 5))

print("Preservation checks passed for all seven pages.")

**Checkpoint:** you now have cleaned, traceable page documents. You can stop here and continue from Step 3 later. Keep your raw/clean comparison: a visually prettier result alone is not sufficient evidence of a better extraction.


## Step 3 - Create and inspect a baseline

**Purpose:** turn cleaned text into chunks and examine what each chunk can actually support.

**Your assignment:** complete `make_chunks(inputs, chunk_size, chunk_overlap, label)` with the demonstrated recursive splitter. Preserve source metadata, give each chunk an ID, and use the supplied page-mapping helper. Start with **800 characters and 100 characters of overlap**.

Inspect chunks about query limits and time predicates. Record (a) one useful chunk and (b) one chunk that needs neighboring context. For each, cite the chunk ID, page, and the missing or preserved information.

**Expected result:** nonempty chunks with accurate page references and two evidence-based observations.

**Connection to Demo 2:** these sizes count Python characters, not words or tokens. The recursive splitter prefers separators such as paragraph breaks and spaces; chunk lengths and actual overlap can vary. Splitting a list of page documents processes each page independently. Even huge overlap cannot reach a different input document.

### Source mapping - supplied

We keep character offsets into the **cleaned page text**, so a repaired cross-page chunk can later name both pages accurately. These offsets are not PDF coordinates or offsets into the raw extraction.

An input is a `Document` with one or more source spans. `annotate_chunk` intersects the chunk's character range with those spans. You use it as supplied infrastructure; interval arithmetic is not a student objective.

In [ ]:
def page_input(page: Document) -> Document:
    """Wrap a cleaned page with a source span covering all of its characters."""
    return Document(page_content=page.page_content, metadata={
        "source": page.metadata["source"],
        "spans": [{"start": 0, "end": len(page.page_content),
                   "pdf_page": page.metadata["pdf_page"],
                   "printed_page": page.metadata["printed_page"], "page_start": 0}],
    })


def annotate_chunk(chunk: Document, original_input: Document, chunk_id: str) -> Document:
    """Map a chunk back to the exact cleaned-page ranges that contributed its text."""
    start = chunk.metadata["start_index"]
    end = start + len(chunk.page_content)
    assert start >= 0 and original_input.page_content[start:end] == chunk.page_content
    spans: list[dict[str, int]] = []
    # Ranges are half-open: start is included, end is excluded.
    for span in original_input.metadata["spans"]:
        lo, hi = max(start, span["start"]), min(end, span["end"])
        # Keep only positive-length intersections with original source text.
        if lo < hi:
            # Translate input-relative offsets back to cleaned-page offsets.
            spans.append({
                "pdf_page": span["pdf_page"], "printed_page": span["printed_page"],
                "start": span["page_start"] + lo - span["start"],
                "end": span["page_start"] + hi - span["start"],
            })
    
    return Document(page_content=chunk.page_content, metadata={
        "source": original_input.metadata["source"], "chunk_id": chunk_id,
        "pdf_pages": sorted({s["pdf_page"] for s in spans}),
        "printed_pages": sorted({s["printed_page"] for s in spans}),
        "source_spans": spans,
    })

In [ ]:
page_inputs: list[Document] = [page_input(p) for p in clean_pages]

### Your code

In [ ]:
# TODO - Step 3.
def make_chunks(inputs: list[Document], chunk_size: int = 800, chunk_overlap: int = 100, label: str = "baseline") -> list[Document]:
    """Split each input independently and attach a unique ID and exact page provenance."""
    # TODO 1: Validate that 0 <= chunk_overlap < chunk_size.
    # TODO 2: Configure RecursiveCharacterTextSplitter with character lengths and start indices.
    # TODO 3: Split each input independently and assign a unique chunk ID using label.
    # TODO 4: Use annotate_chunk to preserve page provenance, then return all chunks.
    # YOUR CODE HERE
    raise NotImplementedError("Step 3: implement make_chunks.")


In [ ]:
baseline_chunks = make_chunks(page_inputs)

assert baseline_chunks and all(c.page_content.strip() for c in baseline_chunks)
assert len({c.metadata["chunk_id"] for c in baseline_chunks}) == len(baseline_chunks)

show_table([{
    "chunk_id": c.metadata["chunk_id"], "pdf_pages": c.metadata["pdf_pages"],
    "characters": len(c.page_content), "preview": c.page_content[:95],
} for c in baseline_chunks])

### Read the evidence - supplied inspection helper


`inspect_chunks` is a literal text locator for a human audit, not a semantic retriever. It prints full matching chunks so you can judge whether the text answers a question on its own. It does not call an embedding model or grade an answer.


In [ ]:
def inspect_chunks(chunks: list[Document], phrase: str) -> list[str]:
    """Print chunks containing a case-insensitive literal phrase and return their IDs."""
    found = [c for c in chunks if phrase.casefold() in c.page_content.casefold()]
    for chunk in found:
        print(f"\n{chunk.metadata['chunk_id']} | PDF pages {chunk.metadata['pdf_pages']}")
        print(chunk.page_content)
    if not found:
        print(f"No complete literal match for {phrase!r}.")
    return [c.metadata["chunk_id"] for c in found]

In [ ]:
_ = inspect_chunks(baseline_chunks, "500")
_ = inspect_chunks(baseline_chunks, "predicate &time")
_ = inspect_chunks(baseline_chunks, "you cannot build a predicate with wildcards (:*) for time")

### Your interpretation


Read the full chunks printed above and record two observations.

1. **A useful chunk:** give its ID and source page(s). Explain which question it can answer and whether it preserves the relevant conditions or caveats.

   *Your response here.*

2. **A chunk that needs neighboring context:** give its ID and source page(s). Identify the missing information and locate it in the original PDF.

   *Your response here.*

For the time-predicate passage, distinguish evidence belonging to that specific passage from a related warning elsewhere in the document.

**Checkpoint:** you now have a baseline ingestion pipeline and observations to investigate in the next comparison.

## Step 4 - Compare three configurations fairly

**Purpose:** observe the tradeoff between evidence staying together, extra context, and repeated material.

**Your assignment:**

1. Keep the source pages, cleaning, splitter class, and questions fixed.
2. Build A: 800/0, B: 800/200, and C: 1200/200 (size/overlap, in characters).
3. Run the supplied inspection audit on the four development cases below.
4. Read at least one flagged case and one apparent success in full. Recommend a configuration and explain its tradeoff.

A versus B isolates overlap. B versus C isolates chunk size. Comparing A directly with C changes two variables.

**Expected result:** a computed comparison table and a short written interpretation. Do not implement retrieval metrics: these questions are a small ingestion inspection checklist, not a golden set for a search engine.

### Four development questions


Answer **according to the supplied 2019 PDF**:

| ID | Inspection question | Source to inspect |
|---|---|---|
| D1 | What quota is stated, and how does it apply to employees behind one proxy/firewall? | PDF page 13 |
| D2 | What are all four steps in the key-registration procedure? | PDF page 13 |
| D3 | Does the specific time-predicate explanation retain both its `&time` introduction and the restriction that continues on the next page? | PDF pages 11–12 |
| D4 | What numeric-range capability is described, and what wildcard restriction accompanies it? | PDF page 11 |

D3 intentionally tests the integrity of a particular passage, even if related information appears elsewhere. “Evidence together” means all listed markers occur in the same chunk. It is a useful flag for manual inspection, not a judgment of completeness or correctness.

In [ ]:
# Supplied markers locate known evidence in this fixed document.

DEV_CASES: list[dict[str, Any]] = [
    {
        "id": "D1", 
        "pages": [13], 
        "markers": ["500 queries per IP", "per day", "proxy server/firewall", "not the individual user"]
    },
    {
        "id": "D2", 
        "pages": [13], 
        "markers": ["1. Go to", "2. Click on", "3. Fill out", "4. You will receive"]
    },
    {
        "id": "D3", 
        "pages": [11, 12], 
        "markers": ["You can limit by time", "predicate &time", "you cannot build a predicate with wildcards (:*) for time"]
    },
    {
        "id": "D4", 
        "pages": [11], 
        "markers": ["range of values for a numeric", "cannot build a predicate with wildcards for numeric"]
    },
]

In [ ]:
def normalized(text: str) -> str:
    """Normalize case and whitespace for marker comparisons, retaining punctuation."""
    return " ".join(text.casefold().split())


def audit_evidence(chunks: list[Document], cases: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Report whether each case has all required pages and text markers in one chunk."""
    rows: list[dict[str, Any]] = []
    for case in cases:
        # Pages and every marker must match the same chunk, not different hits.
        hits = [c for c in chunks
                if set(case["pages"]).issubset(c.metadata["pdf_pages"])
                and all(normalized(m) in normalized(c.page_content) for m in case["markers"])]
        rows.append({"case": case["id"], "evidence_together": bool(hits),
                     "chunk_ids": [c.metadata["chunk_id"] for c in hits],
                     "shortest_hit_chars": min((len(c.page_content) for c in hits), default=None)})
    
    return rows


def chunk_stats(chunks: list[Document]) -> dict[str, int]:
    """Summarize a nonempty chunk list, counting repeated source positions exactly."""
    # Tiny corpus only: offsets let us count actual repeated source characters.
    # This does not assume that requested overlap equals realized overlap.
    covered: set[tuple[int, int]] = set()
    occurrences = 0
    for chunk in chunks:
        for span in chunk.metadata["source_spans"]:
            positions = {(span["pdf_page"], n) for n in range(span["start"], span["end"])}
            # Count each page/offset once here, but every occurrence below.
            covered.update(positions)
            occurrences += len(positions)

    return {
        "chunks": len(chunks),
        "mean_chars": round(sum(len(c.page_content) for c in chunks) / len(chunks)),
        "max_chars": max(len(c.page_content) for c in chunks),
        "repeated_source_chars": occurrences - len(covered),
    }

### Your code

In [ ]:
# TODO - Step 4.
CONFIGS: dict[str, tuple[int, int]] = {"A": (800, 0), "B": (800, 200), "C": (1200, 200)}
variants: dict[str, list[Document]] = {}
comparison_rows: list[dict[str, Any]] = []

# For each configuration:
# 1. Call make_chunks with its size, overlap, and label.
# 2. Store its chunks in variants under the configuration name.
# 3. Run audit_evidence on DEV_CASES.
# 4. Append a comparison row containing the configuration, size, overlap,
#    chunk_stats, and each case ID mapped to its evidence_together result.
# YOUR CODE HERE
raise NotImplementedError("Complete Step 4: build and compare the three configurations.")


In [ ]:
show_table(comparison_rows)
show_table(audit_evidence(variants["C"], DEV_CASES))

### Your comparison and limits of the audit


Use the computed results to answer the following questions.

- What changes between A and B? What changes between B and C?
- Which configuration would you choose, and what tradeoff does it make?
- Does any case remain unresolved across all three configurations? Locate the relevant evidence in the source and explain the result.
- Inspect one flagged case and one apparent success. Do their marker results agree with your reading of the chunks?

*Your comparison and recommendation here.*

Use **C (1200/200)** for Step 5 so you can isolate the effect of its boundary repair, even if your recommendation above differs.

The checks are deliberately limited. Exact markers can miss equivalent phrasing, and a chunk with every marker can still be confusing or contain too much unrelated material. A whole-page chunk may score well without being a useful retrieval unit. Chunk counts and marker hits do not measure future retrieval or answer quality.


## Step 5 - Repair one boundary and defend it

**Purpose:** improve a demonstrated failure without rewriting the source or changing every variable.

**Your assignment:**

1. Keep configuration C (1200/200).
2. On viewer page 11, split just before `Time (Time Series Datasets Only)`.
3. On viewer page 12, split just before `Examples:`. Join the time-section tail from page 11 to the opening continuation from page 12 with a space.
4. Leave the other content in order and run the same splitter again. Use the supplied slicing/joining helpers so provenance remains accurate.
5. Re-run D1-D4, then run the two additional checks below. Inspect the repaired chunk. Write a short decision note describing the gain, its cost, and one remaining limitation.

**Expected result:** a chunk containing the formerly broken time rule, with both source pages in its metadata, and no source-text loss. Do not insert an answer, paraphrase the source, append canned evidence, or drop other sections to improve the checks.

**Why this repair:** a physical page break is not necessarily a meaningful content boundary. We repair one visibly continuous section. This manual intervention is appropriate for learning from a small document; it is not a scalable parser for arbitrary PDFs.

### Slicing and joining with provenance - supplied

`slice_input` keeps an exact text slice and adjusts its source spans. `join_inputs` concatenates slices and shifts their spans. The inserted joining space has no source page. Later, `annotate_chunk` keeps only pages whose actual source characters appear in a chunk. This avoids claiming that every chunk from a two-page input cites both pages.

In [ ]:
def slice_input(doc: Document, start: int, end: int) -> Document:
    """Slice an input using [start, end) offsets and rebase its source spans."""
    if not 0 <= start <= end <= len(doc.page_content):
        raise ValueError("Invalid text slice.")
    spans: list[dict[str, int]] = []
    for old in doc.metadata["spans"]:
        lo, hi = max(start, old["start"]), min(end, old["end"])
        if lo < hi:
            # Rebase the slice; page_start still addresses the original page.
            spans.append({**old, "start": lo - start, "end": hi - start,
                          "page_start": old["page_start"] + lo - old["start"]})
    return Document(page_content=doc.page_content[start:end],
                    metadata={"source": doc.metadata["source"], "spans": spans})


def join_inputs(docs: list[Document], separator: str = " ") -> Document:
    """Join inputs from one source, shifting spans and leaving separators unattributed."""
    if not docs or len({d.metadata["source"] for d in docs}) != 1:
        raise ValueError("Join nonempty slices from the same source.")
    text: str = ""
    spans: list[dict[str, int]] = []
    for i, doc in enumerate(docs):
        if i:
            text += separator
        # Offset includes the separator, which intentionally has no source span.
        offset = len(text)
        spans.extend({**s, "start": s["start"] + offset, "end": s["end"] + offset}
                     for s in doc.metadata["spans"])
        text += doc.page_content
    return Document(page_content=text, metadata={"source": docs[0].metadata["source"], "spans": spans})

### Your code

In [ ]:
# TODO - Step 5.
def repair_time_boundary(inputs: list[Document]) -> list[Document]:
    """Rejoin the known time-rule continuation while retaining all other text in order."""
    # TODO 1: Locate the specified section boundaries on viewer pages 11 and 12.
    # TODO 2: Use slice_input and join_inputs to reunite the time-rule continuation.
    # TODO 3: Return the revised inputs in order, preserving all other source content.
    # YOUR CODE HERE
    raise NotImplementedError("Step 5: implement repair_time_boundary.")

In [ ]:
repaired_inputs = repair_time_boundary(page_inputs)
final_chunks = make_chunks(repaired_inputs, 1200, 200, label="repaired")

show_table(audit_evidence(final_chunks, DEV_CASES))
_ = inspect_chunks(final_chunks, "predicate &time")

### Additional questions and regression checks


Two additional questions check that the repair did not distract us from other useful evidence:

- **V1:** How does the guide illustrate a time range from 1997 to 2012, including its punctuation? (Viewer page 12.)
- **V2:** In the response example, what do the first row and subsequent rows represent? (Viewer page 15.)

These are extra inspection cases, not a statistically meaningful held-out test set. Run them on the baseline C and the repaired version, then read the evidence. If a marker check fails, inspect the text before changing the pipeline.

The supplied structural checks also verify that the repair retains every non-whitespace source character in order and that each reported source slice matches the original cleaned page.


In [ ]:
EXTRA_CASES: list[dict[str, Any]] = [
    {"id": "V1", "pages": [12], "markers": ["&time=from+1997+to+2012"]},
    {"id": "V2", "pages": [15], "markers": [
        "first row provides column names", "subsequent rows provide data values"]},
]

show_table([{"variant": name, **row}
            for name, chunks in [("C", variants["C"]), ("repaired", final_chunks)]
            for row in audit_evidence(chunks, EXTRA_CASES)])

In [ ]:
# Ignore whitespace introduced at joins, but preserve every source character.
before = "".join(d.page_content for d in page_inputs)
after = "".join(d.page_content for d in repaired_inputs)
assert re.sub(r"\s", "", before) == re.sub(r"\s", "", after)

# Each reported span must reproduce its claimed cleaned-page source slice.
for doc in repaired_inputs:
    for s in doc.metadata["spans"]:
        original = clean_by_page[s["pdf_page"]].page_content
        assert doc.page_content[s["start"]:s["end"]] == original[
            s["page_start"]:s["page_start"] + s["end"] - s["start"]]

In [ ]:
# Source spans cover all original page characters exactly once before chunking.
for page in clean_pages:
    number = page.metadata["pdf_page"]
    positions = [n for d in repaired_inputs for s in d.metadata["spans"]
                 if s["pdf_page"] == number
                 for n in range(s["page_start"], s["page_start"] + s["end"] - s["start"])]
    assert sorted(positions) == list(range(len(page.page_content)))

original_d3 = next(r for r in audit_evidence(variants["C"], DEV_CASES) if r["case"] == "D3")
repaired_d3 = next(r for r in audit_evidence(final_chunks, DEV_CASES) if r["case"] == "D3")
assert not original_d3["evidence_together"]
assert repaired_d3["evidence_together"], "Inspect the repaired input and splitter output."
show_table([{"variant": name, **chunk_stats(chunks)}
            for name, chunks in [("C", variants["C"]), ("repaired", final_chunks)]])
print("Boundary and provenance checks passed.")

### Your decision note


Write a **100-150-word decision note** using the results you observed. Address:

- What you changed and which evidence motivated it.
- Whether the repaired chunk preserves the required context and accurate page references.
- What the development and additional checks show, including any remaining failure.
- The cost or tradeoff of the repair and one limitation that remains.

*Your decision note here.*

**Your submission:** the completed notebook, the four-case comparison, two inspected chunks, and a 100-150-word decision note. An alternative repair is acceptable if it preserves content and provenance and explains its results.

## Save your ingestion result - supplied

This optional final cell writes your preprocessing output as ordinary JSON records. You can rerun the pipeline from the supplied PDF whenever needed. Set `SAVE_CHUNKS` to `True` to save the result.

In [ ]:
SAVE_CHUNKS: bool = False  # Set True if you want a local output artifact.

In [ ]:
if SAVE_CHUNKS:
    output_path = Path("outputs/census_chunks.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # Store text and provenance as JSON-compatible records for later reuse.
    records: list[dict[str, Any]] = [{"chunk_id": c.metadata["chunk_id"], "text": c.page_content,
                "metadata": {k: v for k, v in c.metadata.items() if k != "chunk_id"}}
               for c in final_chunks]
    output_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {len(records)} records to {output_path}")

## References


- [Census Data API User Guide v1.6](https://www2.census.gov/api-documentation/api-guide.pdf), Census Bureau, 2019. Original source included unchanged; exercise questions refer to this edition.
- [PyMuPDFLoader](https://reference.langchain.com/python/langchain-community/document_loaders/pdf/PyMuPDFLoader): page-mode loading.
- [Recursive text splitting](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter): separator-based recursive splitting and character lengths.
- [TextSplitter reference](https://reference.langchain.com/python/langchain-text-splitters/base/TextSplitter): `add_start_index` and metadata.


<hr/>